<a href="https://colab.research.google.com/github/MePython313/Chill-pill.chat/blob/main/stable/stable_diffusion_inpainting_webui_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1.

In [ ]:
# CELL 1: INSTALL & LOAD MODEL
print("⚠️ Installing... This takes 3-5 minutes")

!pip install -q diffusers transformers accelerate gradio pillow safetensors

import torch
from diffusers import StableDiffusionInpaintPipeline

if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Go to Runtime > Change runtime type > GPU")

print("✅ GPU found. Loading model...")

pipe = StableDiffusionInpaintPipeline.from_pretrained(
    "runwayml/stable-diffusion-inpainting",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    safety_checker=None,
    requires_safety_checker=False
)
pipe = pipe.to("cuda")
pipe.enable_attention_slicing()

print(f"✅ Model loaded! GPU RAM: {torch.cuda.memory_allocated(0)/1024**3:.2f} GB")
print("👉 Run Cell 2")

2.

In [ ]:
# CELL 2: LAUNCH INTERFACE
import gradio as gr

if 'pipe' not in globals():
    raise RuntimeError("Run Cell 1 first")

def process(image, mask, prompt):
    if image is None:
        return None, "ERROR: No image in Box 1"
    if mask is None:
        return None, "ERROR: No mask in Box 2"
    if not prompt:
        return None, "ERROR: Empty prompt"

    if image.size != mask.size:
        mask = mask.resize(image.size)

    result = pipe(
        prompt=prompt,
        image=image,
        mask_image=mask,
        negative_prompt="ugly, deformed, blurry, low quality",
        num_inference_steps=25,
        guidance_scale=7.5
    ).images[0]

    return result, "SUCCESS! Right-click image to save"

with gr.Blocks() as demo:
    gr.Markdown("# AI Inpainting")
    with gr.Row():
        with gr.Column():
            img = gr.Image(label="1. Original Image", type="pil")
            msk = gr.Image(label="2. Mask (White=Change, Black=Keep)", type="pil")
            prm = gr.Textbox(label="3. Prompt", placeholder="e.g., bare skin, realistic")
            btn = gr.Button("Generate")
        with gr.Column():
            out = gr.Image(label="Result")
            stat = gr.Textbox(label="Status")

    btn.click(fn=process, inputs=[img, msk, prm], outputs=[out, stat])

demo.launch(share=True)

3.

In [ ]:
# CELL 3: CLEANUP
import torch, gc, os, shutil

del pipe
gc.collect()
torch.cuda.empty_cache()

print("✅ Memory cleared")
print("⚠️ Now click: Runtime > Disconnect and delete runtime")